# 🧠 Multi-Tool AI Assistant
### Search • Weather • PDF RAG • Chat History (last 10 messages) • LangSmith Tracing

This notebook builds **one LangChain agent** that can:
1. Search the web for current/latest information (**SerpAPI**)
2. Get the current weather for a city (**OpenWeather**)
3. Answer questions from an uploaded PDF using **RAG** (FAISS + embeddings)

The agent automatically decides which tool to use based on the question.
Only the **latest 10 chat messages** are kept in memory, and every agent/tool
call is traced with **LangSmith**.

---
This version uses **OpenRouter** as the LLM provider (one key gives access to
many models — GPT, Claude, Llama, Gemini, etc. — through a single OpenAI-compatible
API). PDF embeddings run **locally** (`sentence-transformers`, free, no key needed),
so an OpenAI key is not required anywhere in this notebook.

**You will need the following API keys (you'll be prompted securely with `getpass`, nothing is hard-coded):**
| Key | Where to get it |
|---|---|
| `OPENROUTER_API_KEY` (LLM) | https://openrouter.ai/keys |
| `SERPAPI_API_KEY` (Search tool) | https://serpapi.com/manage-api-key |
| `OPENWEATHER_API_KEY` (Weather tool) | https://home.openweathermap.org/api_keys |
| `LANGSMITH_API_KEY` (LangSmith tracing) | https://smith.langchain.com/settings |


## 1. Install Dependencies

In [1]:
# Run once per Colab session
!pip install -q -U langchain langchain-openrouter langchain-community langchain-core \
    faiss-cpu pypdf google-search-results langsmith \
    sentence-transformers

# Re-pin requests to the version Colab's own tools expect, to avoid the
# "google-colab requires requests==X but you have requests==Y" resolver warning
!pip install -q "requests==2.32.4"


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.0 MB/s eta 0:00:00
  

## 2. API Keys (Secure Input — no keys are ever written in code)

Use `getpass` so keys are typed securely and never stored in the notebook file.
In Google Colab you can alternatively use **Colab Secrets** (the 🔑 icon in the
left sidebar) — the fallback code below checks Colab Secrets first, and only
asks via `getpass` if the secret isn't set.


In [2]:
import os
from getpass import getpass

def get_key(env_name: str, prompt: str) -> str:
    """Try Colab Secrets first, then fall back to a secure getpass prompt."""
    # Try Colab userdata secrets (Colab only)
    try:
        from google.colab import userdata
        val = userdata.get(env_name)
        if val:
            os.environ[env_name] = val
            return val
    except Exception:
        pass

    if os.environ.get(env_name):
        return os.environ[env_name]

    val = getpass(f"Enter your {prompt}: ")
    os.environ[env_name] = val
    return val

# --- Required keys ---
OPENROUTER_API_KEY   = get_key("OPENROUTER_API_KEY", "OpenRouter API key (LLM)")
SERPAPI_API_KEY      = get_key("SERPAPI_API_KEY", "SerpAPI key (Search tool)")
OPENWEATHER_API_KEY  = get_key("OPENWEATHER_API_KEY", "OpenWeather API key (Weather tool)")

# --- LangSmith tracing keys ---
LANGSMITH_API_KEY = get_key("LANGSMITH_API_KEY", "LangSmith API key (Tracing)")

print("✅ All keys captured (values are hidden).")


Enter your OpenRouter API key (LLM): ··········
Enter your SerpAPI key (Search tool): ··········
Enter your OpenWeather API key (Weather tool): ··········
Enter your LangSmith API key (Tracing): ··········
✅ All keys captured (values are hidden).


## 3. Enable LangSmith Tracing

In [3]:
import os

# LangSmith tracing configuration (current env-var names)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "beginner-multi-tool-assistant"
# LANGSMITH_API_KEY was already set in the previous cell

print("✅ LangSmith tracing enabled. Project name: 'beginner-multi-tool-assistant'")
print("View traces at: https://smith.langchain.com/")


✅ LangSmith tracing enabled. Project name: 'beginner-multi-tool-assistant'
View traces at: https://smith.langchain.com/


## 4. LLM Setup (via OpenRouter)

We use `langchain-openrouter`'s dedicated `ChatOpenRouter` class rather than
pointing `ChatOpenAI` at a custom `base_url` — it's the purpose-built
integration for OpenRouter and is what's confirmed to work in this environment.
You can switch the `model` string to any model OpenRouter supports
(see https://openrouter.ai/models), e.g. `"openai/gpt-4o-mini"`,
`"anthropic/claude-sonnet-4.5"`, `"google/gemini-2.5-flash"`, etc.


In [4]:
from langchain_openrouter import ChatOpenRouter

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",              # any OpenRouter model id works here
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    max_tokens=500,
)

print("✅ LLM ready via OpenRouter.")


✅ LLM ready via OpenRouter.


## 5. PDF RAG Pipeline
Upload a PDF, split it into chunks, create embeddings (locally, via
`sentence-transformers` — no API key required), store them in a
**FAISS** vector database, and build a retriever.


In [5]:
from google.colab import files

print("📄 Please upload your PDF file (the document you want the assistant to answer questions from).")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_path}")


📄 Please upload your PDF file (the document you want the assistant to answer questions from).


Saving Renewable_Energy_Overview.pdf to Renewable_Energy_Overview.pdf
✅ Uploaded: Renewable_Energy_Overview.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# --- PDF Loader ---
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"✅ Loaded {len(pages)} page(s) from the PDF.")

# --- Split into chunks ---
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)
print(f"✅ Split into {len(chunks)} chunk(s).")

# --- Embeddings (local, free, no API key) + FAISS vector store ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("✅ FAISS vector store created.")

# --- Retriever ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("✅ Retriever ready.")


/tmp/ipykernel_1885/2710986319.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ Loaded 3 page(s) from the PDF.
✅ Split into 5 chunk(s).


/tmp/ipykernel_1885/2710986319.py:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS vector store created.
✅ Retriever ready.


## 6. Tools

Each tool is defined with `@tool` and a **clear description** so the agent can
pick the right one automatically. The PDF RAG tool is built with a modern
**LCEL chain** (`retriever | prompt | llm`) rather than the older
`langchain.chains.RetrievalQA`, since that import path has moved/been
deprecated across recent LangChain versions.


In [7]:
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import requests

# ---------------------------------------------------------------------------
# TOOL 1: Search (SerpAPI) — for latest / current / recent information
# ---------------------------------------------------------------------------
@tool
def search_tool(query: str) -> str:
    """Use this tool to search the web for CURRENT, LATEST, or RECENT information
    that is not likely to be in the PDF or the LLM's own knowledge — for example
    news, live events, or anything time-sensitive. Input should be a plain search query string."""
    if not SERPAPI_API_KEY:
        return "Error: SERPAPI_API_KEY is missing. Please provide a valid SerpAPI key."
    try:
        params = {
            "engine": "google",
            "q": query,
            "api_key": SERPAPI_API_KEY,
        }
        resp = requests.get("https://serpapi.com/search", params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        results = data.get("organic_results", [])
        if not results:
            return "No search results were found for that query."
        snippets = []
        for r in results[:3]:
            title = r.get("title", "")
            snippet = r.get("snippet", "")
            snippets.append(f"- {title}: {snippet}")
        return "\n".join(snippets)
    except requests.exceptions.RequestException as e:
        return f"Error: Search request failed ({e})."
    except Exception as e:
        return f"Error: Unexpected search error ({e})."


# ---------------------------------------------------------------------------
# TOOL 2: Weather (OpenWeather) — for current weather in a city
# ---------------------------------------------------------------------------
@tool
def weather_tool(city: str) -> str:
    """Use this tool to get the CURRENT WEATHER for a given city. Input should be
    a city name, e.g. 'London' or 'Mumbai'. Returns temperature, conditions, and humidity."""
    if not OPENWEATHER_API_KEY:
        return "Error: OPENWEATHER_API_KEY is missing. Please provide a valid OpenWeather key."
    try:
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {"q": city, "appid": OPENWEATHER_API_KEY, "units": "metric"}
        resp = requests.get(url, params=params, timeout=15)
        if resp.status_code == 404:
            return f"Error: '{city}' is not a valid or recognized city name."
        resp.raise_for_status()
        data = resp.json()
        temp = data["main"]["temp"]
        feels_like = data["main"]["feels_like"]
        humidity = data["main"]["humidity"]
        condition = data["weather"][0]["description"]
        return (
            f"Weather in {city}: {condition}, temperature {temp}°C "
            f"(feels like {feels_like}°C), humidity {humidity}%."
        )
    except requests.exceptions.RequestException as e:
        return f"Error: Weather request failed ({e})."
    except Exception as e:
        return f"Error: Unexpected weather error ({e})."


# ---------------------------------------------------------------------------
# TOOL 3: PDF RAG — for questions about the uploaded document
# Built with LCEL (retriever | prompt | llm) instead of the older
# langchain.chains.RetrievalQA, which has moved/been deprecated across
# recent LangChain versions and can raise ModuleNotFoundError depending
# on which sub-packages pip resolves.
# ---------------------------------------------------------------------------
RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are answering questions using only the context below, extracted from "
    "an uploaded PDF. If the answer isn't in the context, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

def format_docs(docs) -> str:
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

@tool
def pdf_rag_tool(question: str) -> str:
    """Use this tool to answer questions about the content of the UPLOADED PDF
    document. Use this whenever the question refers to 'the document', 'the PDF',
    'the file', or asks about specific content that would be inside it."""
    try:
        answer = rag_chain.invoke(question).strip()
        if not answer:
            return "No relevant information was found in the PDF for that question."
        return answer
    except Exception as e:
        return f"Error: Could not retrieve an answer from the PDF ({e})."


tools = [search_tool, weather_tool, pdf_rag_tool]
print("✅ Tools created:", [t.name for t in tools])


✅ Tools created: ['search_tool', 'weather_tool', 'pdf_rag_tool']


## 7. Chat History (Last 10 Messages Only)

We store messages in a simple Python list and use **list slicing** to keep
only the latest 10 message objects (tool calls / tool results also count as
messages).


In [8]:
MAX_HISTORY = 10
chat_history = []  # holds message objects; trimmed to the last MAX_HISTORY after every turn

def show_history():
    print(f"--- Chat history ({len(chat_history)} messages stored) ---")
    for i, m in enumerate(chat_history, 1):
        role = getattr(m, "type", "unknown")
        text = m.content if isinstance(m.content, str) else str(m.content)
        print(f"{i}. [{role}] {text[:80]}")

print("✅ Chat history store ready (max", MAX_HISTORY, "messages).")


✅ Chat history store ready (max 10 messages).


## 8. Agent Creation

One agent, three tools. The **system prompt** tells the LLM exactly when to
use each tool. Built with **`langchain.agents.create_agent`** — the
current high-level agent API in LangChain — since older APIs
(`create_tool_calling_agent` / `AgentExecutor`) have been removed from
recent LangChain releases.


In [9]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are a helpful multi-tool AI assistant. You have access to three tools:

1. search_tool — use this ONLY for questions that need CURRENT, LATEST, or RECENT
   real-world information (e.g. news, today's events, anything time-sensitive).
2. weather_tool — use this for ANY question about current weather in a city.
3. pdf_rag_tool — use this for ANY question about the uploaded PDF document
   (its content, facts inside it, summaries of it, follow-ups about it).

Rules:
- Always pick exactly one relevant tool per question when a tool is clearly needed.
- If the question is a natural follow-up to a previous message (e.g. "what is the
  humidity there?"), use conversation history to resolve what it refers to, and
  call the appropriate tool again with the resolved details.
- If no tool is needed (e.g. general knowledge or chit-chat), answer directly.
- Keep answers concise and clear.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created with tools:", [t.name for t in tools])


✅ Agent created with tools: ['search_tool', 'weather_tool', 'pdf_rag_tool']


In [10]:
def trim_history(messages, max_len=MAX_HISTORY):
    """Keep only the latest max_len messages, but never start the list with
    an orphaned ToolMessage (a tool result whose triggering tool-call message
    got sliced off) — providers reject that as an invalid request."""
    trimmed = list(messages[-max_len:])
    while trimmed and trimmed[0].__class__.__name__ == "ToolMessage":
        trimmed = trimmed[1:]
    return trimmed

def ask_agent(question: str, verbose: bool = True) -> str:
    """Ask the agent while preserving conversation history, keeping only the
    latest MAX_HISTORY message objects (tool calls / tool results count too)."""
    global chat_history

    messages = chat_history + [{"role": "user", "content": question}]

    try:
        # result["messages"] contains the full updated trace: user message,
        # any tool-call / tool-result messages, and the final assistant message.
        result = agent.invoke({"messages": messages})
    except Exception as e:
        # Surface the *actual* provider error instead of just the generic
        # wrapper exception, so you can see exactly what OpenRouter rejected.
        detail = getattr(e, "data", None) or getattr(e, "raw_response", None) or getattr(e, "body", None)
        print(f"⚠️ Agent call failed: {e}")
        if detail:
            print("Provider error detail:", detail)
        return f"Error: the agent call failed ({e})"

    all_messages = result["messages"]

    if verbose:
        for m in all_messages:
            tool_calls = getattr(m, "tool_calls", None)
            if tool_calls:
                for tc in tool_calls:
                    print(f"[agent selected tool] {tc['name']}  args={tc['args']}")
            if m.__class__.__name__ == "ToolMessage":
                print(f"[tool result] {str(m.content)[:200]}")

    # Keep only the latest MAX_HISTORY messages, without orphaning a tool result
    chat_history = trim_history(all_messages, MAX_HISTORY)

    answer = all_messages[-1].content
    print(f"User: {question}")
    print(f"Assistant: {answer}\n")
    return answer

def clear_chat_history() -> None:
    """Delete all stored conversation messages."""
    global chat_history
    chat_history = []
    print("✅ Chat history cleared")

print("✅ ask_agent() helper ready.")


✅ ask_agent() helper ready.


## 9. Test the Agent

Below are the required test questions. `ask_agent()` prints which tool was
selected (if any) and the final answer for each turn. The LangSmith trace
link will appear after tracing is confirmed in Section 10.


### 9.1 — A question requiring current web information (Search tool)

In [11]:
ask_agent("What is the latest news about AI regulation this week?")


[agent selected tool] search_tool  args={'query': 'AI regulation news October 2023'}
[tool result] - CIPL Releases Paper on Ten Recommendations for Global ...: October 10, 2023. CIPL Releases Paper on Ten Recommendations for Global AI Regulation. On September 29, 2023, the Centre for Information Po
User: What is the latest news about AI regulation this week?
Assistant: This week, notable developments in AI regulation include:

1. **CIPL Recommendations**: The Centre for Information Policy Leadership released a paper outlining ten recommendations for global AI regulation, emphasizing the need for a cohesive approach to managing AI technologies.

2. **UK AI Regulatory Updates**: The UK government provided updates on the progress of the AI Act and the AI Liability Directive. Additionally, preparations for the upcoming UK AI Safety Summit were revealed, indicating a focus on establishing safety standards in AI deployment.

These highlights reflect ongoing efforts to create frameworks for r

'This week, notable developments in AI regulation include:\n\n1. **CIPL Recommendations**: The Centre for Information Policy Leadership released a paper outlining ten recommendations for global AI regulation, emphasizing the need for a cohesive approach to managing AI technologies.\n\n2. **UK AI Regulatory Updates**: The UK government provided updates on the progress of the AI Act and the AI Liability Directive. Additionally, preparations for the upcoming UK AI Safety Summit were revealed, indicating a focus on establishing safety standards in AI deployment.\n\nThese highlights reflect ongoing efforts to create frameworks for responsible AI use and governance.'

### 9.2 — Current weather in a city (Weather tool)

In [12]:
ask_agent("What is the current weather in Mumbai?")


[agent selected tool] search_tool  args={'query': 'AI regulation news October 2023'}
[tool result] - CIPL Releases Paper on Ten Recommendations for Global ...: October 10, 2023. CIPL Releases Paper on Ten Recommendations for Global AI Regulation. On September 29, 2023, the Centre for Information Po
[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
User: What is the current weather in Mumbai?
Assistant: The current weather in Mumbai is as follows:
- **Condition**: Broken clouds
- **Temperature**: 29.99°C (feels like 36.01°C)
- **Humidity**: 74%



'The current weather in Mumbai is as follows:\n- **Condition**: Broken clouds\n- **Temperature**: 29.99°C (feels like 36.01°C)\n- **Humidity**: 74%'

### 9.3 — Two questions about the uploaded PDF (PDF RAG tool)

In [13]:
ask_agent("What is this document about?")


[agent selected tool] search_tool  args={'query': 'AI regulation news October 2023'}
[tool result] - CIPL Releases Paper on Ten Recommendations for Global ...: October 10, 2023. CIPL Releases Paper on Ten Recommendations for Global AI Regulation. On September 29, 2023, the Centre for Information Po
[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
[agent selected tool] pdf_rag_tool  args={'question': 'What is this document about?'}
[tool result] This document provides a general overview of renewable energy sources, including solar, wind, hydro, and geothermal power. It discusses how these energy sources work, their benefits and challenges, an
User: What is this document about?
Assistant: The document provides an overview of renewable energy sources, including solar, wind, hydro, and geothermal power. It discusses how these energy sources function, their benefits and challen

'The document provides an overview of renewable energy sources, including solar, wind, hydro, and geothermal power. It discusses how these energy sources function, their benefits and challenges, and their role in reducing global carbon emissions. It serves as sample reference material for testing document question-answering systems.'

In [14]:
ask_agent("What are the main benefits of renewable energy mentioned in the document?")


[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
[agent selected tool] pdf_rag_tool  args={'question': 'What is this document about?'}
[tool result] This document provides a general overview of renewable energy sources, including solar, wind, hydro, and geothermal power. It discusses how these energy sources work, their benefits and challenges, an
[agent selected tool] pdf_rag_tool  args={'question': 'What are the main benefits of renewable energy mentioned in the document?'}
[tool result] The main benefits of renewable energy mentioned in the document are:

1. Reduces greenhouse gas emissions and air pollution.
2. Decreases dependence on imported fossil fuels.
3. Creates jobs in manufa
User: What are the main benefits of renewable energy mentioned in the document?
Assistant: The main benefits of renewable energy mentioned in the document are:

1. Reduces greenhouse gas em

'The main benefits of renewable energy mentioned in the document are:\n\n1. Reduces greenhouse gas emissions and air pollution.\n2. Decreases dependence on imported fossil fuels.\n3. Creates jobs in manufacturing, installation, and maintenance.\n4. Provides long-term price stability since fuel costs are low or zero.'

### 9.4 — Follow-up question (tests conversation context, e.g. resolving 'there')

In [15]:
ask_agent("What is the humidity there?")


[agent selected tool] pdf_rag_tool  args={'question': 'What is this document about?'}
[tool result] This document provides a general overview of renewable energy sources, including solar, wind, hydro, and geothermal power. It discusses how these energy sources work, their benefits and challenges, an
[agent selected tool] pdf_rag_tool  args={'question': 'What are the main benefits of renewable energy mentioned in the document?'}
[tool result] The main benefits of renewable energy mentioned in the document are:

1. Reduces greenhouse gas emissions and air pollution.
2. Decreases dependence on imported fossil fuels.
3. Creates jobs in manufa
[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
User: What is the humidity there?
Assistant: The humidity in Mumbai is currently 74%.



'The humidity in Mumbai is currently 74%.'

### 9.5 — Continue the conversation to show only the latest 10 messages are kept

In [16]:
# Ask a few more questions to push older messages out of history
ask_agent("What is the weather in Delhi?")
ask_agent("What is the weather in Bengaluru?")
ask_agent("Summarize the document in one line.")

show_history()  # should show at most 10 messages, the most recent ones


[agent selected tool] pdf_rag_tool  args={'question': 'What are the main benefits of renewable energy mentioned in the document?'}
[tool result] The main benefits of renewable energy mentioned in the document are:

1. Reduces greenhouse gas emissions and air pollution.
2. Decreases dependence on imported fossil fuels.
3. Creates jobs in manufa
[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
[agent selected tool] weather_tool  args={'city': 'Delhi'}
[tool result] Weather in Delhi: light rain, temperature 34.96°C (feels like 41.96°C), humidity 59%.
User: What is the weather in Delhi?
Assistant: The weather in Delhi is currently light rain, with a temperature of 34.96°C (feels like 41.96°C) and humidity at 59%.

[agent selected tool] weather_tool  args={'city': 'Mumbai'}
[tool result] Weather in Mumbai: broken clouds, temperature 29.99°C (feels like 36.01°C), humidity 74%.
[

### 9.6 — Error-handling checks (missing key / invalid city / unavailable result)

In [17]:
# Invalid city example
print(weather_tool.invoke({"city": "NotARealCityXYZ123"}))

# Missing key simulation (uncomment to test — temporarily clears the key)
# backup_key = SERPAPI_API_KEY
# SERPAPI_API_KEY = ""
# print(search_tool.invoke({"query": "test query"}))
# SERPAPI_API_KEY = backup_key


Error: 'NotARealCityXYZ123' is not a valid or recognized city name.


## 10. Confirm LangSmith Tracing

After running the cells above, open your LangSmith project to confirm the
agent and tool calls were traced.


In [18]:
print("Go to: https://smith.langchain.com/")
print("Open project: 'beginner-multi-tool-assistant'")
print("You should see one trace per ask_agent() call, showing:")
print(" - the agent's reasoning/tool selection step")
print(" - the specific tool that was called (search_tool / weather_tool / pdf_rag_tool)")
print(" - the tool's input and output")
print("Take a screenshot of a trace where a tool was called, to include in your submission.")


Go to: https://smith.langchain.com/
Open project: 'beginner-multi-tool-assistant'
You should see one trace per ask_agent() call, showing:
 - the agent's reasoning/tool selection step
 - the specific tool that was called (search_tool / weather_tool / pdf_rag_tool)
 - the tool's input and output
Take a screenshot of a trace where a tool was called, to include in your submission.
